# Retail Sales Aggregation and Store-Type Comparison
**Cameron Batts**

## Problem
Turn department-level sales into comparable store-week summaries using validated joins and pandas aggregation.

## Approach
Read 421,570 department-week records and 45 store records, exclude holiday observations, validate the many-to-one store join, and sum sales by Store, Date and Type. Compare average store-week sales across store types.

## Results
Recomputed nonholiday averages were $1,369,654.70 for Type A, $816,921.64 for Type B and $472,741.57 for Type C. The overall average was $1,041,256.38 across observed nonholiday store-weeks. These are descriptive differences, not causal effects of store type.

## Limitations
Holiday weeks are excluded, so these figures do not describe full-year sales. Store size, location, department coverage and other confounders are not controlled. Negative sales records are retained. No forecasting, causal estimation or sales uplift was demonstrated.

## Data
Data/sales.csv and Data/stores.csv are included. Dates are day/month/year. The original repository did not document an upstream source or redistribution license; provenance remains unverified.

## Verification
Included-data code cells executed successfully during this update.

In [ ]:

import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name in ('Notebook', 'Notebooks') else Path.cwd()
sales = pd.read_csv(project_root / 'Data' / 'sales.csv')
stores = pd.read_csv(project_root / 'Data' / 'stores.csv')
sales['Date'] = pd.to_datetime(sales['Date'], format='%d/%m/%Y')

In [ ]:
stores.head()

   Store Type    Size
0      1    A  151315
1      2    A  202307
2      3    B   37392
3      4    A  205863
4      5    B   34875


In [ ]:
stores.tail()

    Store Type    Size
40     41    A  196321
41     42    C   39690
42     43    C   41062
43     44    C   39910
44     45    B  118221


In [ ]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Store   45 non-null     int64 
 1   Type    45 non-null     object
 2   Size    45 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.2+ KB


In [ ]:
stores['Type'].value_counts()

Type
A    22
B    17
C     6
Name: count, dtype: int64


We see there are different type of store from A,B, & C. Additionally we see that each store varies in the size of the store.

In [ ]:
sales['IsHoliday'] = sales['IsHoliday'].astype(int)

In [ ]:
sales.query("IsHoliday == 0")

        Store  Dept       Date  Weekly_Sales  IsHoliday
0           1     1 2010-02-05      24924.50          0
2           1     1 2010-02-19      41595.55          0
3           1     1 2010-02-26      19403.54          0
4           1     1 2010-03-05      21827.90          0
5           1     1 2010-03-12      21043.39          0
...       ...   ...        ...           ...        ...
421565     45    98 2012-09-28        508.37          0
421566     45    98 2012-10-05        628.10          0
421567     45    98 2012-10-12       1061.02          0
421568     45    98 2012-10-19        760.01          0
421569     45    98 2012-10-26       1076.80          0

[391909 rows x 5 columns]


In [ ]:
sales_2b = sales.loc[sales['IsHoliday'] == 0]


sales_2b.head()


   Store  Dept       Date  Weekly_Sales  IsHoliday
0      1     1 2010-02-05      24924.50          0
2      1     1 2010-02-19      41595.55          0
3      1     1 2010-02-26      19403.54          0
4      1     1 2010-03-05      21827.90          0
5      1     1 2010-03-12      21043.39          0


Merge stores and sales Used a left-join which attaches store information to each sales row

A left join retains every selected sales record. Validating many-to-one cardinality prevents duplicate store keys from multiplying sales. An inner join would drop only unmatched records; it does not inherently lose rows when all keys match.

In [ ]:
# Merge stores and sales

# Used a left-join which attaches store information to each sales row
sales_stores = sales_2b.merge(stores, on='Store', how='left', validate='many_to_one') 

sales_stores.head()

   Store  Dept       Date  Weekly_Sales  IsHoliday Type    Size
0      1     1 2010-02-05      24924.50          0    A  151315
1      1     1 2010-02-19      41595.55          0    A  151315
2      1     1 2010-02-26      19403.54          0    A  151315
3      1     1 2010-03-05      21827.90          0    A  151315
4      1     1 2010-03-12      21043.39          0    A  151315


In [ ]:

# Create a new dataframe that summarizes weekly sales by Store, Date, and Type
weekly_store_sales = (sales_stores
                    # Group the data by Store, Date, and store Type
                      .groupby(['Store', 'Date', 'Type'])['Weekly_Sales']
                      # For each group, sum the Weekly_Sales values
                      .sum()
                      # Turn the grouped index back into regular columns in a new dataframe
                      .reset_index()
                     )

In [ ]:
print('Average nonholiday store-week sales:', weekly_store_sales['Weekly_Sales'].mean())
print(weekly_store_sales.groupby('Type')['Weekly_Sales'].mean())

Average nonholiday store-week sales: 1041256.3802088555
Type
A    1.369655e+06
B    8.169216e+05
C    4.727416e+05
Name: Weekly_Sales, dtype: float64


**Observations**

Recomputed nonholiday averages were $1,369,654.70 for Type A, $816,921.64 for Type B and $472,741.57 for Type C. The overall average was $1,041,256.38 across observed nonholiday store-weeks. These are descriptive differences, not causal effects of store type. Holiday weeks are excluded, so these figures do not describe full-year sales. Store size, location, department coverage and other confounders are not controlled. Negative sales records are retained. No forecasting, causal estimation or sales uplift was demonstrated.

In [ ]:
Chained_Weekly_Sales = (
    sales
    .query("IsHoliday == 0")                      # filter out holiday weeks
    .merge(stores, on='Store', how='left', validate='many_to_one')       # merge with store info
    .groupby(['Store', 'Date', 'Type'])['Weekly_Sales']
    .sum()                                       # total weekly sales per Store, Date, and Type
    .reset_index()
)

Chained_Weekly_Sales.head()


   Store       Date Type  Weekly_Sales
0      1 2010-02-05    A    1643690.90
1      1 2010-02-19    A    1611968.17
2      1 2010-02-26    A    1409727.59
3      1 2010-03-05    A    1554806.68
4      1 2010-03-12    A    1439541.59
